# 02 — Core Architecture: Layers, Activations & Loss

## What this notebook covers

We now build the **object-oriented skeleton** of our neural network library.
Each component (layer, activation, loss) is a class with `forward()` and `backward()` methods — exactly the design pattern PyTorch and Keras use internally.

| Class | Role |
|---|---|
| `DenseLayer` | Linear transformation: $z = xW + b$ |
| `ReLU` | Non-linearity: $f(x)=\max(0,x)$ |
| `Softmax` | Output probabilities |
| `SoftmaxWithCrossEntropy` | Fused output + loss (faster backward) |

---

## Why fuse Softmax + CrossEntropy?

Taken separately, the Softmax backward requires computing a full Jacobian per sample.
When fused with CrossEntropy the math cancels out beautifully:

$$\frac{\partial L}{\partial z_i} = \hat{y}_i - y_i$$

Just subtract 1 at the position of the true class, then normalise. That's it.


## 1. Imports and setup

In [ ]:
import sys
sys.path.append('..')   # so we can import from core/

import numpy as np
import nnfs
from nnfs.datasets import spiral_data
nnfs.init()

from core import DenseLayer, ReLU, SoftmaxWithCrossEntropy

X, y = spiral_data(samples=100, classes=3)
print(f'Dataset shape — X: {X.shape}, y: {y.shape}')


## 2. Inspect the spiral dataset

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='brg', s=20)
plt.title('Spiral Dataset — 3 classes')
plt.xlabel('x₁'); plt.ylabel('x₂')
plt.grid(True); plt.tight_layout(); plt.show()


## 3. One complete forward + backward pass

We're not training yet — just verifying that gradients flow correctly end-to-end.


In [ ]:
# Build the network
dense1         = DenseLayer(2, 3)
activation1    = ReLU()
dense2         = DenseLayer(3, 3)
loss_fn        = SoftmaxWithCrossEntropy()

# --- Forward pass ---
dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
loss = loss_fn.forward(dense2.output, y)

preds    = np.argmax(loss_fn.output, axis=1)
accuracy = np.mean(preds == y)
print(f'Initial loss:     {loss:.4f}')
print(f'Initial accuracy: {accuracy:.4f}  (random ≈ 0.333)')

# --- Backward pass ---
loss_fn.backward(loss_fn.output, y)
dense2.backward(loss_fn.dinputs)
activation1.backward(dense2.dinputs)
dense1.backward(activation1.dinputs)

print('\nGradient shapes (should match weight shapes):')
print(f'  dense1.dweights: {dense1.dweights.shape}')
print(f'  dense2.dweights: {dense2.dweights.shape}')


## 4. Key takeaways

- Every class has a **`forward()`** (compute output) and **`backward()`** (compute gradients)
- Gradients flow **right to left**: loss → dense2 → relu → dense1
- Each layer receives `dinputs` from the layer to its right and passes `dinputs` to the layer to its left
- The fused `SoftmaxWithCrossEntropy` backward is just: `y_pred - y_true`, normalised

➡️ Next notebook: add an optimiser and actually train the network.
